In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pytz

In [2]:
from API.get_data.api_yahoo import get_historical_data, get_analyst_price_targets, get_business_recommendation, get_business_grades
from utils.old_trends import get_trends_events

In [3]:
from utils.trends import get_trends_events
from utils.old_slope import qualify_slope

In [4]:
df = get_historical_data("MSFT", "5y")

In [5]:
df2=df.copy()

In [6]:
today = datetime.now()#.strftime('%Y-%m-%d')
first_of_this_month = today.replace(day=1)
first_of_last_month = first_of_this_month - relativedelta(months=1)

# Remettre l'heure à zéro (minuit)
first_of_this_month_midnight = first_of_this_month.replace(hour=0, minute=0, second=0, microsecond=0)
first_of_last_month_midnight = first_of_last_month.replace(hour=0, minute=0, second=0, microsecond=0)

one_year_rolling = today - relativedelta(months=12)
one_year_rolling_midnight = one_year_rolling.replace(hour=0, minute=0, second=0, microsecond=0)

five_year_rolling = today - relativedelta(year=5)
five_year_rolling_midnight = five_year_rolling.replace(hour=0, minute=0, second=0, microsecond=0)

year_to_date_midnight = datetime(first_of_this_month.year, 1, 1, 0, 0)

# Reset date to LocalTime 
df["Date"] = df["Date"].apply(lambda x: x.tz_localize(None))

# filter Dataframe
this_month_data = df[df["Date"]>=first_of_this_month_midnight]
last_month_data = df[df["Date"]>=first_of_last_month_midnight]
year_to_date_data = df[df["Date"]>=year_to_date_midnight]
one_year_rolling_period_data = df[df["Date"]>=one_year_rolling_midnight]
five_year_rolling_period_data = df[df["Date"]>=five_year_rolling_midnight]

tendance sur  
- mois   
- year to date   
- N-1  
- 5 ans  
- 10 ans   

In [7]:
different_dataframe_filter_bydate = [this_month_data, last_month_data, year_to_date_data, one_year_rolling_period_data, five_year_rolling_period_data]

In [8]:
from utils.trends_slope import check_trend_of_price

In [9]:
for df_filter in different_dataframe_filter_bydate:
    check_trend_of_price(df_filter)

c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["EMA"] = df["Close"].ewm(span=ema_period, adjust=False).mean()
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["EMA_Slope"] = (
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:59: Setting

In [10]:
# this_month_data, ==> Very bullish, UP
# last_month_data, ==> Very bullish, UP
# year_to_date_data, ==> Bullish, UP
# one_year_rolling_period_data, ==> Stagnant, UP
# five_year_rolling_period_data ==> Stagnant, UP


In [15]:
a

{'Close_slope': 0.1962902727254965}

In [ ]:
for

In [16]:
for df_filter in different_dataframe_filter_bydate:
    a = get_trends_events(
        df=df_filter,
        cols=["Close"],
        # cols = ["Open", "Low", "High", "Close"],
        degre=3)

    for col in a :
        pred = qualify_slope(a.get(col))
        print(f"Col : {col} with value = {a.get(col):2f} have been {pred["steepness"]}")
    print ("______________\n")

Col : Close_slope with value = 3.395004 have been Very bullish
______________

Col : Close_slope with value = 1.547894 have been Very bullish
______________

Col : Close_slope with value = 0.574778 have been Bullish
______________

Col : Close_slope with value = 0.350378 have been Bullish
______________

Col : Close_slope with value = 0.204570 have been Bullish
______________



c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends.py:14: RankWarning: Polyfit may be poorly conditioned
  coefficients = np.polyfit(x, y, degre)


In [12]:
from graph.trends_line_open_high_low_close import get_trends_events_graph

In [13]:
def calculate_slope(series):
    # series.index donne les index, ici nous voulons des valeurs numériques 0, 1, 2...
    # pour représenter le temps.
    x = np.arange(len(series))
    if len(x) < 2: # Nécessite au moins 2 points pour une pente
        return np.nan
    # polyfit renvoie [pente, ordonnée_à_l_origine]
    slope, _ = np.polyfit(x, series, 1)
    return slope

def get_slope(df, ema_period=5, slope_window=5):
    df['EMA'] = df['Close'].ewm(span=ema_period, adjust=False).mean()
    # Appliquer la fonction de pente sur une fenêtre glissante de l'EMA
    df['EMA_Slope'] = df['EMA'].rolling(window=slope_window, min_periods=2).apply(calculate_slope, raw=False)

    # Déterminer la direction basée sur la pente
    df['EMA_Direction_Slope'] = np.select(
        [df['EMA_Slope'] > 0.001, df['EMA_Slope'] < -0.001], # Utiliser un seuil pour considérer 'plat'
        ['Up', 'Down'],
        default='Flat' # Considéré comme plat si la pente est très proche de zéro
    )
    # --- Visualisation de la pente ---
    plt.figure(figsize=(14, 10))
    ax2 = plt.subplot(2, 1, 2) # Partage l'axe X pour le zoom synchronisé
    ax2.plot(df['EMA_Slope'], label='Pente de l\'EMA', color='blue')
    ax2.axhline(0, color='grey', linestyle='--', linewidth=0.8) # Ligne zéro pour référence
    ax2.set_title('Pente de l\'EMA')
    ax2.legend()
    ax2.grid(True)

    # Ajouter des zones colorées pour la direction
    ax2.fill_between(df.index, 0, df['EMA_Slope'], where=df['EMA_Slope'] > 0.001, color='lightgreen', alpha=0.5, label='Up Trend')
    ax2.fill_between(df.index, 0, df['EMA_Slope'], where=df['EMA_Slope'] < -0.001, color='lightcoral', alpha=0.5, label='Down Trend')

    plt.tight_layout()
    plt.show()

In [14]:
for df_filter in different_dataframe_filter_bydate:
    get_slope(df_filter,5,5)

C:\Users\cleme\AppData\Local\Temp\ipykernel_26368\1305114582.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['EMA'] = df['Close'].ewm(span=ema_period, adjust=False).mean()
C:\Users\cleme\AppData\Local\Temp\ipykernel_26368\1305114582.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['EMA_Slope'] = df['EMA'].rolling(window=slope_window, min_periods=2).apply(calculate_slope, raw=False)
C:\Users\cleme\AppData\Local\Temp\ipykernel_26368\1305114582.py:17: SettingWithCopyWarning: 
A value is trying t

NameError: name 'plt' is not defined

In [ ]:
for df_filter in different_dataframe_filter_bydate:
    get_trends_events_graph(
    df=df_filter,
    cols = ["Open", "Low", "High", "Close"],
    degre=2,
    show_curve=True,
    colors = {
        "Open": "blue",
        "Low": "red",
        "High": "green",
        "Close": "orange"
    }
)

In [ ]:
get_trends_events_graph(
    df=df,
    cols = ["Open", "Low", "High", "Close"],
    degre=2,
    show_curve=True,
    colors = {
        "Open": "blue",
        "Low": "red",
        "High": "green",
        "Close": "orange"
    }
)

{'Open_slope': np.float64(0.19600416103634274),
 'Low_slope': np.float64(0.1948105389758358),
 'High_slope': np.float64(0.19740980260843036),
 'Close_slope': np.float64(0.19629027582068231)}

In [ ]:
get_trends_events(
    x=df.index.values, y=df["Open"],
    degre=6,
    show_curve=True)

TypeError: get_trends_events() got an unexpected keyword argument 'x'

In [ ]:
get_trends_events(
    x=df.index.values, y=df["Low"],
    degre=6,
    show_curve=True)

In [ ]:
from utils.trends_candles import candlestick_price_chart

In [ ]:
get_trends_events(
    df=df,
    cols = ["Open", "Low", "High", "Close"],
    degre=6,
    show_curve=True,
    colors = {
        "Open": "blue",
        "Low": "red",
        "High": "green",
        "Close": "orange"
    }
)

{'Open_slope': np.float64(0.17495127380681993),
 'Low_slope': np.float64(0.17256229142348112),
 'High_slope': np.float64(0.17750345326503578),
 'Close_slope': np.float64(0.1749589990003311)}

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
# Tracé
df.set_index('Date', inplace=True)
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Open'], label='Open', color='blue')
plt.plot(df.index, df['High'], label='High', color='green')
plt.plot(df.index, df['Low'], label='Low', color='red')
plt.plot(df.index, df['Close'], label='Close', color='orange')

plt.title("Évolution des prix")
plt.xlabel("Date")
plt.ylabel("Prix")

In [ ]:
df

---

In [ ]:
# fonction a rajouter a get info company preporcessing -- info business

In [ ]:
period        sell  strongBuy  strongSell symbol
2025-04-01     9          9           3   TSLA

In [ ]:
9/(9+9+3)

In [ ]:
9/(7+16+14+9+2)

In [ ]:
import numpy as np
def u(df_trend):
    df_trend["sell"] = df_trend["sell"] / total_recommendation
    df_trend["strongBuy"] = df_trend["strongBuy"] / total_recommendation
    df_trend["strongSell"] = df_trend["strongSell"] / total_recommendation
    mean_sell = np.round(df_trend["sell"].mean(),2)
    mean_strongBuy = np.round(df_trend["strongBuy"].mean(),2)
    mean_strongSell = np.round(df_trend["strongSell"].mean(),2)

In [ ]:
get_business_recommendation("TSLA")

In [ ]:
get_business_grades("AAPL")

In [ ]:
get_analyst_price_targets("AAPL")